# Pump.fun Market Lab 概览

适合读者：
- 想要查看公开 Pump.fun 市场数据样本的 builder 和研究者。

前置条件：
- 基础 Python 和 pandas 使用经验。
- 本仓库的本地 checkout。

学习目标：
- 读取小时级 manifest。
- 验证北京时间完整一天的数据覆盖。
- 对比 `pump_fun` 和 `pump_amm` 活跃度。
- 在不下载完整 release 数据的情况下读取小型 parquet 样本。


## 大纲

1. 读取 manifest。
2. 按市场汇总行数和文件大小。
3. 查看小型 parquet 样本。
4. 对比小时级活跃度。
5. 练习：找到最繁忙的北京时间小时。


In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
MANIFEST = ROOT / "data_manifest" / "pumpfun_market_2026-07-31_manifest.csv"
SAMPLE_DIR = ROOT / "data_sample"

assert MANIFEST.exists(), MANIFEST
assert SAMPLE_DIR.exists(), SAMPLE_DIR


## 1. 读取小时级 manifest

manifest 是公开数据集的轻量索引，记录每个预期小时文件、所属市场、行数、大小和 schema。


In [ ]:
manifest = pd.read_csv(MANIFEST)
manifest["beijing_hour"] = pd.to_datetime(manifest["beijing_hour"])
manifest["utc_hour"] = pd.to_datetime(manifest["utc_hour"])
manifest.head()


## 2. 覆盖范围与规模

北京时间完整一天中，每个市场都应该有 24 个小时文件。


In [ ]:
coverage = (
    manifest.groupby("dataset")
    .agg(
        hours=("exists", "sum"),
        rows=("rows", "sum"),
        size_gib=("size_bytes", lambda s: s.sum() / 1024**3),
    )
    .reset_index()
)
coverage


## 3. 查看小型 parquet 样本

完整一天的数据适合通过 release assets 分发，因为它太大，不适合放进普通 Git 历史。本仓库保留了小型 parquet 样本，所以只 clone 仓库也能跑通演示流程。


In [ ]:
sample_files = sorted(SAMPLE_DIR.glob("*.parquet"))
[(p.name, round(p.stat().st_size / 1024, 1)) for p in sample_files]


In [ ]:
for path in sample_files:
    table = pq.read_table(path)
    print(path.name, table.num_rows, "rows", len(table.column_names), "columns")
    print(table.column_names[:8], "...")


## 4. 小时级活跃度对比

在这个样本日里，AMM 市场的原始事件量明显更高。这不代表更容易盈利，只描述市场活跃度。


In [ ]:
hourly = manifest.pivot_table(
    index=manifest["beijing_hour"].dt.hour,
    columns="dataset",
    values="rows",
    aggfunc="sum",
)
hourly.head()


In [ ]:
# 纯文本图表：即使没有 matplotlib 也能稳定显示。
scaled = hourly / hourly.max()
for hour, row in scaled.iterrows():
    fun_bar = "#" * int(row.get("pump_fun", 0) * 24)
    amm_bar = "#" * int(row.get("pump_amm", 0) * 24)
    print(f"{hour:02d}: pump_fun {fun_bar:<24} | pump_amm {amm_bar:<24}")


## 5. 练习

找出每个市场最繁忙的北京时间小时，然后查看 manifest 中对应的行。


In [ ]:
# 练习答案框架
busiest = manifest.loc[manifest.groupby("dataset")["rows"].idxmax(), [
    "dataset", "beijing_hour", "rows", "size_bytes"
]].copy()
busiest["size_mib"] = busiest["size_bytes"] / 1024**2
busiest.drop(columns=["size_bytes"])


## 常见误区

不要把行数当成交易优势。高活跃小时在计入滑点、优先费、平台费、失败交易和基础设施成本后，仍然可能很难交易。

## 扩展

下载完整 release assets 后，可以继续使用同一个 manifest CSV，并读取 `path` 列中列出的原始小时级 parquet 文件。
